In [ ]:
import pandas as pd
import numpy as np
import os
import time
from sklearn.utils.class_weight import compute_class_weight

import cudf
import dask_cudf
import dask.array as da

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import (
    classification_report, accuracy_score, balanced_accuracy_score,
    f1_score, precision_score, recall_score, matthews_corrcoef,
    cohen_kappa_score, confusion_matrix
)

from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

import warnings
warnings.filterwarnings('ignore')
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


In [ ]:
import random
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

In [ ]:
class HybridDataset(Dataset):
    def __init__(self, X_seq, X_static, y):
        self.X_seq = torch.FloatTensor(X_seq)
        self.X_static = torch.FloatTensor(X_static)
        self.y = torch.LongTensor(y)

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        return self.X_seq[idx], self.X_static[idx], self.y[idx]


class HybridLSTMModel(nn.Module):
    def __init__(self, input_dim_per_phase, static_dim):
        super().__init__()

        # ===== LSTM branch =====
        self.hidden_dim = 128
        self.num_layers = 1
        self.dropout_p = 0.3
        self.num_classes = 3

        self.lstm = nn.LSTM(
            input_size=input_dim_per_phase,
            hidden_size=self.hidden_dim,
            num_layers=self.num_layers,
            batch_first=True
        )

        self.dropout = nn.Dropout(self.dropout_p)

        # ===== Static branch (MLP) =====
        self.mlp = nn.Sequential(
            nn.Linear(static_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU()
        )

        # ===== Fusion layer =====
        self.fc = nn.Linear(self.hidden_dim + 32, self.num_classes)

    def forward(self, x_seq, x_static):
        # ----- LSTM branch -----
        out, (h_n, c_n) = self.lstm(x_seq)

        # Lấy last hidden state theo đúng yêu cầu
        last_hidden = h_n[-1]

        last_hidden = self.dropout(last_hidden)

        # ----- MLP branch -----
        static_out = self.mlp(x_static)

        # ----- Combine -----
        combined = torch.cat([last_hidden, static_out], dim=1)

        return self.fc(combined)


In [ ]:
# ===================== METRICS =====================
def gmean_score(y_true, y_pred):
    cm = confusion_matrix(y_true, y_pred)
    per_class = []
    for i in range(cm.shape[0]):
        tp = cm[i,i]
        fn = cm[i].sum() - tp
        fp = cm[:,i].sum() - tp
        tn = cm.sum() - tp - fn - fp
        sens = tp / (tp + fn) if (tp + fn) > 0 else 0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0
        per_class.append(np.sqrt(sens * spec))
    return np.prod(per_class) ** (1/len(per_class)) if per_class else 0


def gmean_per_class(y_true, y_pred, target_class):
    cm = confusion_matrix(y_true, y_pred)
    i = target_class
    tp = cm[i,i]
    fn = cm[i].sum() - tp
    fp = cm[:,i].sum() - tp
    tn = cm.sum() - tp - fn - fp
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    return np.sqrt(recall * specificity)

# ===================== PRINT RESULTS =====================
def print_results(version_name, phase, y_true, y_pred, time_build_model=None, time_predict=None):
    target_names = ['Excellent', 'Good', 'Average']

    print(f"\n{'='*30} {version_name} - Phase {phase} {'='*30}")
    print(classification_report(y_true, y_pred, digits=10, target_names=target_names))

    prec = precision_score(y_true, y_pred, average=None)
    rec = recall_score(y_true, y_pred, average=None)
    f1 = f1_score(y_true, y_pred, average=None)

    gmeans = [gmean_per_class(y_true, y_pred, i) for i in range(3)]

    print("G-Mean per class (one-vs-rest):")
    for i, name in enumerate(target_names):
        print(f"  {name:<10}: {gmeans[i]:.10f}")

    metrics = {
        'Version': version_name,
        'Phase': phase,
        'TimeBuildModel': time_build_model,
        'TimePredict': time_predict,
        'Accuracy': accuracy_score(y_true, y_pred),
        'BalancedAcc': balanced_accuracy_score(y_true, y_pred),
        'Precision Macro': precision_score(y_true, y_pred, average='macro'),
        'Precision Weighted': precision_score(y_true, y_pred, average='weighted'),
        'Recall Macro': recall_score(y_true, y_pred, average='macro'),
        'Recall Weighted': recall_score(y_true, y_pred, average='weighted'),
        'F1-Score Macro': f1_score(y_true, y_pred, average='macro'),
        'F1-Score Weighted': f1_score(y_true, y_pred, average='weighted'),
        'GMean': gmean_score(y_true, y_pred),
        'MCC': matthews_corrcoef(y_true, y_pred),
        'Kappa': cohen_kappa_score(y_true, y_pred),
    }

    for i, name in enumerate(target_names):
        metrics[f'Precision_{name}'] = prec[i]
        metrics[f'Recall_{name}'] = rec[i]
        metrics[f'F1-Score_{name}'] = f1[i]
        metrics[f'G-Mean_{name}'] = gmeans[i]

    for k, v in metrics.items():
        if k not in ['Version','Phase'] and v is not None:
            print(f"{k:22} : {v:.10f}")

    return metrics


# Hàm chuẩn bị dữ liệu và train

In [ ]:
# ===================== TRAIN =====================
def prepare_and_train_hybrid(train_path, val_path, device, version_name):

    print(f"Loading train (GPU): {train_path}")
    # Load tập train bằng dask_cudf
    ddf_train = dask_cudf.read_parquet(train_path)

    # Load tập valid bằng pandas (RAM)
    df_val = pd.read_parquet(val_path, engine='pyarrow') if val_path else None

    # Đếm số lượng mẫu (Dask cần .compute() hoặc len() trên dask_cudf)
    train_len = len(ddf_train)
    print(f"Train samples: {train_len}")
    if df_val is not None:
        print(f"Validation samples: {len(df_val)}")

    # Loại bỏ cột bằng dask_cudf
    cols_to_drop = ['user_id', 'course_id']
    ddf_train = ddf_train.drop(columns=[c for c in cols_to_drop if c in ddf_train.columns])

    if df_val is not None:
        df_val = df_val.drop(columns=cols_to_drop, errors='ignore')

    # Tách y và X cho tập Train
    # Chuyển về numpy để đưa vào PyTorch Dataset (Dask -> CuDF -> Numpy)
    y_train = ddf_train['label_3'].compute().to_numpy()
    X_train_ddf = ddf_train.drop('label_3', axis=1)

    if df_val is not None:
        y_val = df_val['label_3'].values
        X_val_df = df_val.drop('label_3', axis=1)

    # Xác định các cột (Dùng columns của dask_cudf)
    train_columns = X_train_ddf.columns.tolist()
    phase_cols = [c for c in train_columns if any(f"_p{p}_" in c for p in ['1','2','3','4'])]
    static_cols = [c for c in train_columns if c not in phase_cols]

    for p in ['1','2','3','4']:
        print(f"Phase {p}: {len([c for c in phase_cols if f'_p{p}_' in c])} features")

    # Hàm build_seq xử lý trên Dask/GPU
    def build_seq_dask(ddf, p_cols):
        phases = []
        for p in ['1','2','3','4']:
            cols = sorted([c for c in p_cols if f"_p{p}_" in c])
            # Chuyển từng phase về numpy
            phases.append(ddf[cols].compute().to_numpy())
        # Stack lại thành (N, T, F)
        return np.stack(phases, axis=1)

    # Hàm build_seq cho Pandas (Valid)
    def build_seq_pandas(df, p_cols):
        phases = []
        for p in ['1','2','3','4']:
            cols = sorted([c for c in p_cols if f"_p{p}_" in c])
            phases.append(df[cols].values)
        return np.stack(phases, axis=1)

    # Thực hiện build sequence
    X_seq_train = build_seq_dask(X_train_ddf, phase_cols)
    X_static_train = X_train_ddf[static_cols].compute().to_numpy()

    print(f"Time-series shape: {X_seq_train.shape}")
    print(f"Static feature shape: {X_static_train.shape}")

    # ---Tiền xử lý & Train---
    scaler_seq = StandardScaler()
    N, T, F = X_seq_train.shape
    X_seq_train = scaler_seq.fit_transform(X_seq_train.reshape(-1, F)).reshape(N, T, F)

    scaler_static = StandardScaler()
    X_static_train = scaler_static.fit_transform(X_static_train)

    if df_val is not None:
        X_seq_val = build_seq_pandas(X_val_df, phase_cols)
        X_static_val = X_val_df[static_cols].values
        N2 = X_seq_val.shape[0]
        X_seq_val = scaler_seq.transform(X_seq_val.reshape(-1, F)).reshape(N2, T, F)
        X_static_val = scaler_static.transform(X_static_val)

    le = LabelEncoder()
    y_train_enc = le.fit_transform(y_train)
    print(f"Classes: {le.classes_}")

    if df_val is not None:
        y_val_enc = le.transform(y_val)

    train_loader = DataLoader(HybridDataset(X_seq_train, X_static_train, y_train_enc), batch_size=256, shuffle=True)

    if df_val is not None:
        val_loader = DataLoader(HybridDataset(X_seq_val, X_static_val, y_val_enc), batch_size=256, shuffle=False)

    model = HybridLSTMModel(F, X_static_train.shape[1]).to(device)

    # Class weights
    unique_classes = np.unique(y_train_enc)
    weights = compute_class_weight('balanced', classes=unique_classes, y=y_train_enc)
    weights = torch.FloatTensor(weights).to(device)

    criterion = nn.CrossEntropyLoss(weight=weights)
    optimizer = optim.Adam(model.parameters(), lr=0.001)

    # Vòng lặp Training
    best_loss = float('inf')
    patience = 10
    wait = 0
    start_train = time.perf_counter()

    for epoch in range(50):
        model.train()
        train_loss = 0
        for xb_seq, xb_static, yb in train_loader:
            xb_seq, xb_static, yb = xb_seq.to(device), xb_static.to(device), yb.to(device)
            optimizer.zero_grad()
            out = model(xb_seq, xb_static)
            loss = criterion(out, yb)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        train_loss /= len(train_loader)

        if df_val is not None:
            model.eval()
            val_loss = 0
            with torch.no_grad():
                for xb_seq, xb_static, yb in val_loader:
                    xb_seq, xb_static, yb = xb_seq.to(device), xb_static.to(device), yb.to(device)
                    val_loss += criterion(model(xb_seq, xb_static), yb).item()
            val_loss /= len(val_loader)
            print(f"Epoch {epoch+1}: train = {train_loss:.4f}, val = {val_loss:.4f}")
            monitor = val_loss
        else:
            print(f"Epoch {epoch+1}: loss = {train_loss:.4f}")
            monitor = train_loss

        if monitor < best_loss - 1e-4:
            best_loss = monitor
            best_state = model.state_dict()
            wait = 0
        else:
            wait += 1
            if wait >= patience: break

    model.load_state_dict(best_state)
    time_build = time.perf_counter() - start_train

    os.makedirs("saved_models", exist_ok=True)
    torch.save(model.state_dict(), f"saved_models/LSTM_{version_name}.pt")

    return model, scaler_seq, scaler_static, le, phase_cols, static_cols, time_build

# Chạy từng version

In [ ]:
def run_experiment(base_path, train_file, val_file, test_prefix, version_name, device=None):
    if device is None:
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

    print(f"\n{'#'*20}")
    print(f"Version: {version_name}")
    print(f"{'#'*20}")

    # 1. Huấn luyện mô hình
    model, scaler_seq, scaler_static, le, phase_cols, static_cols, time_build = \
        prepare_and_train_hybrid(f"{base_path_1}/{train_file}", f"{base_path}/{val_file}", device, version_name)

    results = []

    # Chuyển model sang chế độ eval một lần duy nhất trước khi test
    model.eval()

    # 2. Vòng lặp test qua 4 phase
    for phase in range(1, 5):
        test_path = f"{base_path}/{test_prefix}_{phase}.parquet"
        print(f"\n--- Test Phase {phase}: {test_path} ---")

        # Load tập test bằng Pandas
        df = pd.read_parquet(test_path, engine='pyarrow')
        df = df.drop(columns=['user_id','course_id'], errors='ignore')

        y_test_raw = df['label_3'].values
        X_df = df.drop('label_3', axis=1)

        # Hàm build sequence (Giống như cũ nhưng xử lý gọn hơn)
        def build_seq_local(df_input):
            phases_list = []
            for p in ['1','2','3','4']:
                cols = sorted([c for c in phase_cols if f"_p{p}_" in c])
                phases_list.append(df_input[cols].values)
            return np.stack(phases_list, axis=1)

        X_seq_test = build_seq_local(X_df)
        X_static_test = X_df[static_cols].values

        # Transform dữ liệu
        N, T, F = X_seq_test.shape
        X_seq_test = scaler_seq.transform(X_seq_test.reshape(-1, F)).reshape(N, T, F)
        X_static_test = scaler_static.transform(X_static_test)

        # Mã hóa nhãn thực tế
        y_test_enc = le.transform(y_test_raw)

        # 3. CHIA BATCH CHO INFERENCE (Khắc phục lỗi OOM)
        test_dataset = HybridDataset(X_seq_test, X_static_test, y_test_enc)
        test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False)

        all_probs = []
        start_time = time.perf_counter()

        with torch.no_grad():
            for xb_seq, xb_static, _ in test_loader:
                # Đưa từng batch nhỏ lên GPU
                xb_seq = xb_seq.to(device)
                xb_static = xb_static.to(device)

                outputs = model(xb_seq, xb_static)

                # Tính xác suất và đưa về CPU ngay lập tức để giải phóng VRAM
                prob = torch.softmax(outputs, dim=1).cpu().numpy()
                all_probs.append(prob)

        time_pred = time.perf_counter() - start_time

        # Gộp các batch lại thành mảng lớn trên CPU/RAM
        probs = np.vstack(all_probs)
        preds = np.argmax(probs, axis=1)

        # 4. Tính toán và in kết quả
        metrics = print_results(version_name, phase, y_test_enc, preds, time_build, time_pred)

        # 5. Lưu kết quả
        os.makedirs("results_LSTM", exist_ok=True)

        # Lưu Confusion Matrix
        cm = confusion_matrix(y_test_enc, preds)
        pd.DataFrame(cm).to_csv(f"results_LSTM/confusion_matrix_{version_name}_phase{phase}.csv", index=False)

        # Lưu Ma trận xác suất
        df_prob = pd.DataFrame(probs, columns=[f"Prob_Class_{i}" for i in range(probs.shape[1])])
        df_prob['y_true'] = y_test_enc
        df_prob['y_pred'] = preds
        df_prob.to_csv(f"results_LSTM/probability_matrix_{version_name}_phase{phase}.csv", index=False)

        results.append(metrics)

        # Giải phóng bộ nhớ tạm sau mỗi phase test
        del df, X_seq_test, X_static_test, all_probs, probs, preds
        torch.cuda.empty_cache()

    # Tổng hợp bảng kết quả cuối cùng
    df_results = pd.DataFrame(results).round(10)

    ordered_cols = [
        "Version","Phase","TimeBuildModel","TimePredict","Accuracy","BalancedAcc",
        "Precision Macro","Precision Weighted","Recall Macro","Recall Weighted",
        "F1-Score Macro","F1-Score Weighted","GMean","MCC","Kappa",
        "Precision_Excellent","Recall_Excellent","F1-Score_Excellent","G-Mean_Excellent",
        "Precision_Good","Recall_Good","F1-Score_Good","G-Mean_Good",
        "Precision_Average","Recall_Average","F1-Score_Average","G-Mean_Average"
    ]

    df_results = df_results[[c for c in ordered_cols if c in df_results.columns]]

    return df_results

## V_Median

In [ ]:
base_path = "/kaggle/input/datasets/anhtran10/lo-dataset/Median/Median"

In [ ]:
df_v1 = run_experiment(
    base_path=base_path,
    train_file="train_median.parquet",
    val_file="val.parquet",
    test_prefix="test",
    version_name="V1 (Median)"
)
df_v1


####################
Version: V1 (Median)
####################
Loading train (GPU): /kaggle/input/datasets/anhtran10/lo-dataset/Median/Median/train_median.parquet
Train samples: 1859619
Validation samples: 232452
Phase 1: 39 features
Phase 2: 39 features
Phase 3: 39 features
Phase 4: 39 features
Time-series shape: (1859619, 4, 39)
Static feature shape: (1859619, 23)
Classes: [0 1 2]
Epoch 1: train = 0.2421, val = 0.1441
Epoch 2: train = 0.1615, val = 0.1295
Epoch 3: train = 0.1372, val = 0.1228
Epoch 4: train = 0.1232, val = 0.1068
Epoch 5: train = 0.1183, val = 0.1086
Epoch 6: train = 0.1104, val = 0.1220
Epoch 7: train = 0.1022, val = 0.1096
Epoch 8: train = 0.0958, val = 0.1014
Epoch 9: train = 0.0967, val = 0.1244
Epoch 10: train = 0.0892, val = 0.1250
Epoch 11: train = 0.0857, val = 0.1290
Epoch 12: train = 0.0849, val = 0.1268
Epoch 13: train = 0.0786, val = 0.1260
Epoch 14: train = 0.0814, val = 0.1308
Epoch 15: train = 0.0789, val = 0.1254
Epoch 16: train = 0.0766, val = 0.137

,Version,Phase,TimeBuildModel,TimePredict,Accuracy,BalancedAcc,Precision Macro,Precision Weighted,Recall Macro,Recall Weighted,F1-Score Macro,F1-Score Weighted,GMean,MCC,Kappa,Precision_Excellent,Recall_Excellent,F1-Score_Excellent,G-Mean_Excellent,Precision_Good,Recall_Good,F1-Score_Good,G-Mean_Good,Precision_Average,Recall_Average,F1-Score_Average,G-Mean_Average
0,V1 (Median),1,764.245006,3.430408,0.831553,0.812957,0.358551,0.996335,0.812957,0.831553,0.350157,0.904986,0.867473,0.114360,0.029776,0.063218,0.838565,0.117573,0.910252,0.012632,0.768595,0.024855,0.805048,0.999803,0.831711,0.908043,0.890809
1,V1 (Median),2,764.245006,3.801524,0.833304,0.840637,0.355305,0.996359,0.840637,0.833304,0.344925,0.905930,0.883848,0.119218,0.031254,0.052134,0.865471,0.098344,0.923252,0.013947,0.823140,0.027430,0.835548,0.999834,0.833300,0.909002,0.895039
2,V1 (Median),3,764.245006,3.530926,0.889147,0.906841,0.362695,0.996483,0.906841,0.889147,0.369330,0.938135,0.930840,0.158344,0.051668,0.064927,0.937220,0.121441,0.961807,0.023236,0.894215,0.045295,0.898054,0.999922,0.889088,0.941254,0.933760
3,V1 (Median),4,764.245006,3.475201,0.992041,0.948489,0.519712,0.997381,0.948489,0.992041,0.615727,0.994144,0.965496,0.529658,0.453906,0.222455,0.950673,0.360544,0.973468,0.336829,0.902479,0.490566,0.947784,0.999852,0.992315,0.996069,0.975483


In [ ]:
df_v1.to_csv("results_v1.csv", index=False)

## V_SMOTE

In [ ]:
base_path_1 = "/kaggle/input/datasets/uyentran10/lo-smote-test"

In [ ]:
base_path = "/kaggle/input/datasets/anhtran10/lo-dataset/Median/Median"

In [ ]:
df_v22 = run_experiment(
    base_path=base_path,
    train_file="train_median_smote.parquet",
    val_file="val.parquet",
    test_prefix="test",
    version_name="V22 (Median SMOTE)"
)
df_v22


####################
Version: V22 (Median SMOTE)
####################
Loading train (GPU): /kaggle/input/datasets/uyentran10/lo-smote-test/train_median_smote.parquet
Train samples: 5558991
Validation samples: 232452
Phase 1: 39 features
Phase 2: 39 features
Phase 3: 39 features
Phase 4: 39 features
Time-series shape: (5558991, 4, 39)
Static feature shape: (5558991, 23)
Classes: [0 1 2]
Epoch 1: train = 0.0514, val = 0.0312
Epoch 2: train = 0.0253, val = 0.0267
Epoch 3: train = 0.0210, val = 0.0268
Epoch 4: train = 0.0188, val = 0.0295
Epoch 5: train = 0.0174, val = 0.0248
Epoch 6: train = 0.0164, val = 0.0253
Epoch 7: train = 0.0155, val = 0.0254
Epoch 8: train = 0.0149, val = 0.0261
Epoch 9: train = 0.0145, val = 0.0233
Epoch 10: train = 0.0140, val = 0.0271
Epoch 11: train = 0.0137, val = 0.0259
Epoch 12: train = 0.0134, val = 0.0257
Epoch 13: train = 0.0131, val = 0.0259
Epoch 14: train = 0.0128, val = 0.0258
Epoch 15: train = 0.0127, val = 0.0254
Epoch 16: train = 0.0125, val = 0.

,Version,Phase,TimeBuildModel,TimePredict,Accuracy,BalancedAcc,Precision Macro,Precision Weighted,Recall Macro,Recall Weighted,F1-Score Macro,F1-Score Weighted,GMean,MCC,Kappa,Precision_Excellent,Recall_Excellent,F1-Score_Excellent,G-Mean_Excellent,Precision_Good,Recall_Good,F1-Score_Good,G-Mean_Good,Precision_Average,Recall_Average,F1-Score_Average,G-Mean_Average
0,V22 (Median SMOTE),1,2170.202251,3.409260,0.995556,0.492980,0.742835,0.996338,0.492980,0.995556,0.439676,0.994624,0.273360,0.205985,0.197303,0.231441,0.475336,0.311307,0.688924,1.000000,0.004959,0.009868,0.070418,0.997065,0.998644,0.997854,0.421065
1,V22 (Median SMOTE),2,2170.202251,3.642848,0.992037,0.594448,0.581589,0.995594,0.594448,0.992037,0.407989,0.992802,0.347371,0.184870,0.177889,0.122363,0.780269,0.211550,0.880952,0.625000,0.008264,0.016313,0.090909,0.997403,0.994811,0.996105,0.523386
2,V22 (Median SMOTE),3,2170.202251,3.496599,0.988574,0.685910,0.541951,0.995795,0.685910,0.988574,0.452599,0.991492,0.597746,0.215467,0.186699,0.091224,0.946188,0.166404,0.968310,0.536765,0.120661,0.197031,0.347316,0.997865,0.990882,0.994361,0.635054
3,V22 (Median SMOTE),4,2170.202251,3.642655,0.995066,0.902766,0.591902,0.997463,0.902766,0.995066,0.687737,0.995977,0.929351,0.593135,0.553541,0.361314,0.887892,0.513619,0.941570,0.414796,0.824793,0.551991,0.906801,0.999597,0.995614,0.997601,0.940100


In [ ]:
df_v22.to_csv("results_v22.csv", index=False)

## V_GAN

In [ ]:
base_path_1 = "/kaggle/input/datasets/anhtran10/lo-gan-test"

In [ ]:
df_v23 = run_experiment(
    base_path=base_path,
    train_file="train_median_gan.parquet",
    val_file="val.parquet",
    test_prefix="test",
    version_name="V23 (Median GAN)"
)
df_v23


####################
Version: V23 (Median GAN)
####################
Loading train (GPU): /kaggle/input/datasets/anhtran10/lo-gan-test/train_median_gan.parquet
Train samples: 5558991
Validation samples: 232452
Phase 1: 39 features
Phase 2: 39 features
Phase 3: 39 features
Phase 4: 39 features
Time-series shape: (5558991, 4, 39)
Static feature shape: (5558991, 23)
Classes: [0 1 2]
Epoch 1: train = 0.0037, val = 0.0059
Epoch 2: train = 0.0021, val = 0.0055
Epoch 3: train = 0.0019, val = 0.0050
Epoch 4: train = 0.0018, val = 0.0052
Epoch 5: train = 0.0017, val = 0.0047
Epoch 6: train = 0.0016, val = 0.0050
Epoch 7: train = 0.0016, val = 0.0047
Epoch 8: train = 0.0015, val = 0.0048
Epoch 9: train = 0.0015, val = 0.0050
Epoch 10: train = 0.0014, val = 0.0048
Epoch 11: train = 0.0014, val = 0.0051
Epoch 12: train = 0.0014, val = 0.0052
Epoch 13: train = 0.0014, val = 0.0051
Epoch 14: train = 0.0013, val = 0.0050
Epoch 15: train = 0.0013, val = 0.0050

--- Test Phase 1: /kaggle/input/datasets

,Version,Phase,TimeBuildModel,TimePredict,Accuracy,BalancedAcc,Precision Macro,Precision Weighted,Recall Macro,Recall Weighted,F1-Score Macro,F1-Score Weighted,GMean,MCC,Kappa,Precision_Excellent,Recall_Excellent,F1-Score_Excellent,G-Mean_Excellent,Precision_Good,Recall_Good,F1-Score_Good,G-Mean_Good,Precision_Average,Recall_Average,F1-Score_Average,G-Mean_Average
0,V23 (Median GAN),1,1724.811024,3.691287,0.953186,0.523766,0.677259,0.996295,0.523766,0.953186,0.358052,0.973138,0.433778,0.146676,0.073422,1.000000,0.017937,0.035242,0.133930,0.032969,0.598347,0.062495,0.755609,0.998808,0.955013,0.976420,0.806546
1,V23 (Median GAN),2,1724.811024,3.573293,0.920134,0.602080,0.674781,0.996767,0.602080,0.920134,0.398076,0.955598,0.618685,0.139728,0.053938,1.000000,0.103139,0.186992,0.321153,0.025041,0.781818,0.048528,0.848362,0.999302,0.921282,0.958708,0.869193
2,V23 (Median GAN),3,1724.811024,3.632637,0.984156,0.697936,0.681096,0.996935,0.697936,0.984156,0.566924,0.989755,0.769775,0.325310,0.242820,0.928571,0.349776,0.508143,0.591411,0.115414,0.758678,0.200349,0.864387,0.999304,0.985356,0.992281,0.892266
3,V23 (Median GAN),4,1724.811024,3.436419,0.998529,0.836405,0.890293,0.998424,0.836405,0.998529,0.860444,0.998454,0.862237,0.779059,0.776618,0.873874,0.869955,0.871910,0.932658,0.797938,0.639669,0.710092,0.799624,0.999068,0.999590,0.999329,0.859551


In [ ]:
df_v23.to_csv("results_v23.csv", index=False)

## V_CDSMOTE

In [ ]:
df_v2 = run_experiment(
    base_path=base_path,
    train_file="train_median_cdsmote.parquet",
    val_file="val.parquet",
    test_prefix="test",
    version_name="V2 (Median CDS)"
)
df_v2


####################
Version: V2 (Median CDS)
####################
Loading train (GPU): /kaggle/input/datasets/anhtran10/lo-dataset/Median/Median/train_median_cdsmote.parquet
Train samples: 5558987
Validation samples: 232452
Phase 1: 39 features
Phase 2: 39 features
Phase 3: 39 features
Phase 4: 39 features
Time-series shape: (5558987, 4, 39)
Static feature shape: (5558987, 23)
Classes: [0 1 2]
Epoch 1: train = 0.0378, val = 0.0270
Epoch 2: train = 0.0187, val = 0.0243
Epoch 3: train = 0.0157, val = 0.0256
Epoch 4: train = 0.0141, val = 0.0188
Epoch 5: train = 0.0130, val = 0.0207
Epoch 6: train = 0.0122, val = 0.0216
Epoch 7: train = 0.0116, val = 0.0204
Epoch 8: train = 0.0112, val = 0.0224
Epoch 9: train = 0.0108, val = 0.0196
Epoch 10: train = 0.0105, val = 0.0222
Epoch 11: train = 0.0101, val = 0.0210
Epoch 12: train = 0.0100, val = 0.0221
Epoch 13: train = 0.0097, val = 0.0243
Epoch 14: train = 0.0096, val = 0.0208

--- Test Phase 1: /kaggle/input/datasets/anhtran10/lo-dataset/M

,Version,Phase,TimeBuildModel,TimePredict,Accuracy,BalancedAcc,Precision Macro,Precision Weighted,Recall Macro,Recall Weighted,F1-Score Macro,F1-Score Weighted,GMean,MCC,Kappa,Precision_Excellent,Recall_Excellent,F1-Score_Excellent,G-Mean_Excellent,Precision_Good,Recall_Good,F1-Score_Good,G-Mean_Good,Precision_Average,Recall_Average,F1-Score_Average,G-Mean_Average
0,V2 (Median CDS),1,1776.978326,3.879127,0.994756,0.468099,0.651337,0.995000,0.468099,0.994756,0.496841,0.994694,0.463552,0.238401,0.238264,0.760000,0.170404,0.278388,0.412789,0.196699,0.236364,0.214715,0.485560,0.997311,0.997530,0.997421,0.496964
1,V2 (Median CDS),2,1776.978326,3.701097,0.995315,0.582376,0.636595,0.995567,0.582376,0.995315,0.601649,0.995415,0.612530,0.356835,0.356718,0.644295,0.430493,0.516129,0.656045,0.267684,0.319008,0.291101,0.564165,0.997806,0.997625,0.997716,0.620931
2,V2 (Median CDS),3,1776.978326,3.870178,0.986234,0.745216,0.432238,0.995577,0.745216,0.986234,0.477504,0.990500,0.763624,0.276192,0.220675,0.088602,0.829596,0.160104,0.907082,0.209611,0.418182,0.279249,0.645338,0.998503,0.987868,0.993157,0.760686
3,V2 (Median CDS),4,1776.978326,3.565846,0.995836,0.890779,0.619009,0.997550,0.890779,0.995836,0.709378,0.996489,0.919509,0.617091,0.588673,0.400402,0.892377,0.552778,0.944051,0.457088,0.783471,0.577345,0.884064,0.999537,0.996490,0.998011,0.931513


In [ ]:
df_v2.to_csv("results_v2.csv", index=False)

## V_SMOTified GAN

In [ ]:
base_path_1 = "/kaggle/input/datasets/uyentran10/lo-smotifiedgan-test"

In [ ]:
df_v24 = run_experiment(
    base_path=base_path,
    train_file="train_median_smotified_gan.parquet",
    val_file="val.parquet",
    test_prefix="test",
    version_name="V24 (Median SMOTified GAN)"
)
df_v24


####################
Version: V24 (Median SMOTified GAN)
####################
Loading train (GPU): /kaggle/input/datasets/uyentran10/lo-smotifiedgan-test/train_median_smotified_gan.parquet
Train samples: 5558991
Validation samples: 232452
Phase 1: 39 features
Phase 2: 39 features
Phase 3: 39 features
Phase 4: 39 features
Time-series shape: (5558991, 4, 39)
Static feature shape: (5558991, 23)
Classes: [0 1 2]
Epoch 1: train = 0.0038, val = 0.0059
Epoch 2: train = 0.0021, val = 0.0055
Epoch 3: train = 0.0019, val = 0.0052
Epoch 4: train = 0.0018, val = 0.0052
Epoch 5: train = 0.0017, val = 0.0050
Epoch 6: train = 0.0016, val = 0.0049
Epoch 7: train = 0.0016, val = 0.0048
Epoch 8: train = 0.0015, val = 0.0051
Epoch 9: train = 0.0015, val = 0.0051
Epoch 10: train = 0.0014, val = 0.0048
Epoch 11: train = 0.0014, val = 0.0049
Epoch 12: train = 0.0014, val = 0.0048
Epoch 13: train = 0.0014, val = 0.0049
Epoch 14: train = 0.0013, val = 0.0050
Epoch 15: train = 0.0013, val = 0.0052
Epoch 16: t

,Version,Phase,TimeBuildModel,TimePredict,Accuracy,BalancedAcc,Precision Macro,Precision Weighted,Recall Macro,Recall Weighted,F1-Score Macro,F1-Score Weighted,GMean,MCC,Kappa,Precision_Excellent,Recall_Excellent,F1-Score_Excellent,G-Mean_Excellent,Precision_Good,Recall_Good,F1-Score_Good,G-Mean_Good,Precision_Average,Recall_Average,F1-Score_Average,G-Mean_Average
0,V24 (Median SMOTified GAN),1,1841.63287,3.401690,0.995354,0.387063,0.496127,0.994172,0.387063,0.995354,0.397023,0.994628,0.238901,0.166001,0.158998,0.285714,0.008969,0.017391,0.094702,0.205752,0.153719,0.175970,0.391766,0.996914,0.998502,0.997707,0.367509
1,V24 (Median SMOTified GAN),2,1841.63287,3.877781,0.996154,0.387104,0.551816,0.994502,0.387104,0.996154,0.415274,0.995086,0.278311,0.198157,0.166450,0.304348,0.031390,0.056911,0.177167,0.354260,0.130579,0.190821,0.361244,0.996839,0.999344,0.998090,0.336826
2,V24 (Median SMOTified GAN),3,1841.63287,3.438148,0.995930,0.446403,0.668354,0.995224,0.446403,0.995930,0.481854,0.995349,0.419114,0.286471,0.276249,0.677419,0.094170,0.165354,0.306865,0.330377,0.246281,0.282197,0.495944,0.997267,0.998757,0.998011,0.483744
3,V24 (Median SMOTified GAN),4,1841.63287,3.688493,0.998555,0.833976,0.898199,0.998449,0.833976,0.998555,0.862867,0.998477,0.860424,0.782011,0.779101,0.888889,0.860987,0.874715,0.927846,0.806653,0.641322,0.714549,0.800665,0.999055,0.999620,0.999337,0.857454


In [ ]:
df_v24.to_csv("results_v24.csv", index=False)

## V_CDSGAN

In [ ]:
base_path_1 = "/kaggle/input/datasets/hngliththu/cdsmote-gan"

In [ ]:
base_path = "/kaggle/input/datasets/anhtran10/lo-dataset/Median/Median"

In [ ]:
df_v13 = run_experiment(
    base_path=base_path,
    train_file="train_median_cdsmote_gan.parquet",
    val_file="val.parquet",
    test_prefix="test",
    version_name="V13 (Median CDSMOTified GAN)"
)
df_v13


####################
Version: V13 (Median CDSMOTified GAN)
####################
Loading train (GPU): /kaggle/input/datasets/hngliththu/cdsmote-gan/train_median_cdsmote_gan.parquet
Train samples: 5558987
Validation samples: 232452
Phase 1: 39 features
Phase 2: 39 features
Phase 3: 39 features
Phase 4: 39 features
Time-series shape: (5558987, 4, 39)
Static feature shape: (5558987, 23)
Classes: [0 1 2]
Epoch 1: train = 0.0038, val = 0.0058
Epoch 2: train = 0.0020, val = 0.0053
Epoch 3: train = 0.0018, val = 0.0048
Epoch 4: train = 0.0017, val = 0.0046
Epoch 5: train = 0.0016, val = 0.0047
Epoch 6: train = 0.0016, val = 0.0046
Epoch 7: train = 0.0015, val = 0.0048
Epoch 8: train = 0.0015, val = 0.0047
Epoch 9: train = 0.0014, val = 0.0046
Epoch 10: train = 0.0014, val = 0.0047
Epoch 11: train = 0.0014, val = 0.0048
Epoch 12: train = 0.0013, val = 0.0049
Epoch 13: train = 0.0013, val = 0.0048
Epoch 14: train = 0.0013, val = 0.0051

--- Test Phase 1: /kaggle/input/datasets/anhtran10/lo-data

,Version,Phase,TimeBuildModel,TimePredict,Accuracy,BalancedAcc,Precision Macro,Precision Weighted,Recall Macro,Recall Weighted,F1-Score Macro,F1-Score Weighted,GMean,MCC,Kappa,Precision_Excellent,Recall_Excellent,F1-Score_Excellent,G-Mean_Excellent,Precision_Good,Recall_Good,F1-Score_Good,G-Mean_Good,Precision_Average,Recall_Average,F1-Score_Average,G-Mean_Average
0,V13 (Median CDSMOTified GAN),1,1546.274063,3.287063,0.986436,0.481836,0.509806,0.994988,0.481836,0.986436,0.411416,0.990279,0.459867,0.188149,0.155602,0.448276,0.058296,0.103175,0.241437,0.083247,0.398347,0.137714,0.627525,0.997896,0.988866,0.993360,0.641894
1,V13 (Median CDSMOTified GAN),2,1546.274063,3.577977,0.991594,0.513105,0.633265,0.995610,0.513105,0.991594,0.475633,0.993207,0.536897,0.269268,0.250201,0.756757,0.125561,0.215385,0.354338,0.144977,0.419835,0.215528,0.645850,0.998062,0.993921,0.995987,0.676275
2,V13 (Median CDSMOTified GAN),3,1546.274063,3.211194,0.995423,0.630077,0.641388,0.996068,0.630077,0.995423,0.628519,0.995707,0.681674,0.424020,0.421739,0.622754,0.466368,0.533333,0.682819,0.303173,0.426446,0.354396,0.652193,0.998237,0.997418,0.997827,0.711292
3,V13 (Median CDSMOTified GAN),4,1546.274063,3.510946,0.998400,0.847042,0.856258,0.998337,0.847042,0.998400,0.850423,0.998360,0.873681,0.767595,0.767154,0.820084,0.878924,0.848485,0.937423,0.749533,0.662810,0.703509,0.813896,0.999158,0.999391,0.999275,0.874087


In [ ]:
df_v13.to_csv("results_v13.csv", index=False)